# VRSBench Image Captioning: QLoRA fine-tuning of Qwen3-VL-8B on Kaggle (8 h budget)

**Inter-IIT internal hackathon, Task 01.** Everything here is self-contained and reproducible.

**What this notebook does**
1. Downloads VRSBench (`xiang709/VRSBench`) and builds the caption train set from the **train** split.
2. Picks **50 fixed samples** from the official validation caption file (`VRSBench_EVAL_Cap.json`). No leakage: train/eval image ids are checked for overlap.
3. Implements **BERT-BLEU4** exactly as written in the reference report: `LP * exp(1/4 * sum_n log P_n)`, `P_n` = mean over reference n-grams of the max cosine to candidate n-grams, `LP = exp(-0.5 * |Lc - Lr| / Lr)`. Token-level BERT (`bert-base-uncased`) is the headline score, MiniLM phrase-level is the secondary score.
4. Evaluates the **zero-shot baseline** on the 50 samples, trains with **QLoRA (Unsloth)** under a **hard 8 h wall-clock cutoff**, and re-evaluates.
5. Saves the adapter, predictions, loss curve, and a `results.json` with throughput / memory / metric numbers for your PPT.

**Optimisations used (all measured and logged)**
- 4-bit NF4 base weights (QLoRA) + LoRA on vision encoder, attention and MLP
- Unsloth gradient checkpointing, fused kernels, 8-bit AdamW
- fp16 on T4/P100, bf16 automatically on Ampere+
- Loss on the **caption tokens only** (prompt + image tokens are masked)
- **Time-aware cosine LR schedule**: the LR decays against `max(elapsed/budget, step/total_steps)`, so the schedule finishes exactly at the 8 h cutoff instead of being chopped mid-way
- Lazy image loading (no 15 GB of PIL images in RAM), 2 dataloader workers
- Checkpoints every N steps with elapsed-time bookkeeping, so a crashed session can resume without losing the budget accounting
- Optional micro-benchmark (`run_microbench=True`) that compares gradient checkpointing / batch size / optimizer on measured step time and peak VRAM

**Kaggle settings:** Accelerator = GPU (T4 x2 or P100; only GPU 0 is used), **Internet = On**. Use *Save Version, Save & Run All (Commit)* so it runs in the background (12 h session limit; total here is about 8 h train + 30 to 50 min setup/eval).

In [1]:
!pip install -q -U unsloth unsloth_zoo
!pip install -q hf_transfer sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.4/230.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216

## 1. Imports and configuration

In [2]:
import os, time, json, math, random, re, shutil, zipfile, gc, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"            # Unsloth is single-GPU; avoids Trainer DataParallel on 2xT4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
NB_T0 = time.time()

from unsloth import FastVisionModel, is_bf16_supported      # must be imported before transformers / trl
from unsloth.trainer import UnslothVisionDataCollator
import torch, numpy as np
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from types import SimpleNamespace as NS

CFG = NS(
    model_name = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",   # faster alternative: ...Qwen3-VL-4B-Instruct-unsloth-bnb-4bit
    train_budget_sec = 8 * 3600,      # HARD cutoff for training wall-clock
    seed = 3407,
    n_eval = 50,
    max_train_samples = None,         # None = all caption samples in the train split
    epochs_cap = 3,                   # upper bound; the time budget normally stops training first
    per_device_bs = 4,
    grad_accum = 4,                   # effective batch = 16
    lr = 2e-4,
    min_lr_ratio = 0.05,
    warmup_steps = 30,
    lora_r = 32, lora_alpha = 32, lora_vision = True,
    optim = "adamw_8bit",
    gc_mode = "unsloth",              # Unsloth gradient checkpointing
    max_len = 1024,
    num_workers = 2,                  # set 0 if you hit dataloader/fork errors
    save_steps = 200,
    resume = True,                    # resume from newest checkpoint (budget accounting is restored)
    train_on_responses_only = True,
    run_microbench = False,           # optional ~15 min ablation, see section 4
    decode_sweep = True,              # compare greedy vs beam search on the 50 eval samples
    prompt = "Provide a detailed caption for this remote sensing image.",
)

random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)

def free_gb(p):
    try: return shutil.disk_usage(p).free / 2**30
    except Exception: return -1
cands = [p for p in ["/kaggle/temp", "/tmp"] if os.path.exists(p)]
DATA_ROOT = max(cands, key=free_gb)
DATA_DIR  = os.path.join(DATA_ROOT, "vrsbench"); HF_CACHE = os.path.join(DATA_ROOT, "hf_cache")
OUT_DIR   = "/kaggle/working/vrsbench_run" if os.path.exists("/kaggle/working") else "./vrsbench_run"
for d in (DATA_DIR, HF_CACHE, OUT_DIR): os.makedirs(d, exist_ok=True)
print(f"data root: {DATA_ROOT} ({free_gb(DATA_ROOT):.0f} GB free) | outputs: {OUT_DIR}")
if free_gb(DATA_ROOT) < 25: print("WARNING: less than 25 GB free; downloads of the two zips need about 12 GB plus extraction.")
print(torch.cuda.get_device_name(0), "| bf16 supported:", is_bf16_supported())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
data root: /tmp (1119 GB free) | outputs: /kaggle/working/vrsbench_run
Tesla T4 | bf16 supported: False


## 2. Data: VRSBench captions (train split) and 50 fixed eval samples (validation split)

The HF repo layout (confirmed): `Images_train.zip`, `Images_val.zip`, `VRSBench_train.json` (LLaVA format, mixed tasks), `VRSBench_EVAL_Cap.json` (validation captions).
The first cell prints raw records so you can verify the field names; the parser is tolerant to small naming differences.

In [3]:
from huggingface_hub import hf_hub_download, list_repo_files
REPO = "xiang709/VRSBench"
print(list_repo_files(REPO, repo_type="dataset"))
def dl(fn): return hf_hub_download(REPO, fn, repo_type="dataset", cache_dir=HF_CACHE)
def base(x): return os.path.basename(str(x))
def stem(x): return os.path.splitext(base(x))[0]
def first(d, keys):
    for k in keys:
        if k in d and d[k]: return d[k]
    return None

# ---------------- eval set: official validation captions ----------------
eval_raw = json.load(open(dl("VRSBench_EVAL_Cap.json")))
if isinstance(eval_raw, dict): eval_raw = list(eval_raw.values())
print("EVAL_Cap records:", len(eval_raw)); print("sample record:", json.dumps(eval_raw[0], indent=1)[:700])
eval_items, seen = [], set()
for it in eval_raw:
    img = first(it, ["image_id", "image", "img"]); ref = first(it, ["ground_truth", "answer", "caption", "gt", "text"])
    if img and ref and base(img) not in seen:
        seen.add(base(img)); eval_items.append(dict(image=base(img), ref=str(ref).strip()))
eval_samples = random.Random(CFG.seed).sample(eval_items, CFG.n_eval)
print("unique eval images:", len(eval_items), "| using:", len(eval_samples))

# ---------------- train set: caption entries from the LLaVA-format train json ----------------
train_raw = json.load(open(dl("VRSBench_train.json")))
print("train json records:", len(train_raw)); print("sample record:", json.dumps(train_raw[0], indent=1)[:900])
eval_stems = {stem(s["image"]) for s in eval_items}
train_pairs, seen_tr = [], set()
for it in train_raw:
    conv = it.get("conversations") or []
    if len(conv) < 2: continue
    q, a = str(conv[0].get("value", "")), str(conv[1].get("value", "")).strip()
    t = str(it.get("type", it.get("task", ""))).lower()
    is_cap = ("cap" in t) or re.search(r"\[caption\]|describe the image|detailed (caption|description)", q, re.I)
    img = first(it, ["image", "image_id", "img"])
    if is_cap and a and img and stem(img) not in seen_tr:
        seen_tr.add(stem(img)); train_pairs.append((base(img), a))

if len(train_pairs) < 1000:      # fallback: per-image annotation jsons
    print("Caption filter found few items -> falling back to Annotations_train.zip")
    with zipfile.ZipFile(dl("Annotations_train.zip")) as z:
        for n in z.namelist():
            if n.endswith(".json"):
                d = json.loads(z.read(n)); cap = d.get("caption")
                if cap and stem(n) not in seen_tr:
                    seen_tr.add(stem(n)); train_pairs.append((base(d.get("image_id") or (stem(n) + ".png")), cap.strip()))
assert len(train_pairs) > 1000, "Could not parse captions from VRSBench_train.json - inspect the printed record above and adapt the parser."
leak = {stem(p[0]) for p in train_pairs} & eval_stems
assert not leak, f"train/eval overlap: {len(leak)} images"
if CFG.max_train_samples: train_pairs = train_pairs[:CFG.max_train_samples]
print("caption training pairs:", len(train_pairs), "| train/eval overlap: 0")

# ---------------- download zips, extract only the images we need, delete zips ----------------
def extract_needed(zip_path, out_dir, wanted):
    os.makedirs(out_dir, exist_ok=True); found = {}
    with zipfile.ZipFile(zip_path) as z:
        for info in tqdm(z.infolist(), desc=os.path.basename(zip_path)):
            if info.is_dir(): continue
            b = base(info.filename)
            if b in wanted or stem(b) in wanted:
                dst = os.path.join(out_dir, b)
                if not os.path.exists(dst):
                    with z.open(info) as src, open(dst, "wb") as f: shutil.copyfileobj(src, f)
                found[b] = dst; found[stem(b)] = dst
    return found

t0 = time.time()
val_map = extract_needed(dl("Images_val.zip"), os.path.join(DATA_DIR, "val"), {s["image"] for s in eval_samples} | {stem(s["image"]) for s in eval_samples})
tr_map  = extract_needed(dl("Images_train.zip"), os.path.join(DATA_DIR, "train"), {p[0] for p in train_pairs} | {stem(p[0]) for p in train_pairs})
shutil.rmtree(HF_CACHE, ignore_errors=True)       # free the ~12 GB of zips
for s in eval_samples: s["path"] = val_map.get(s["image"]) or val_map.get(stem(s["image"]))
eval_samples = [s for s in eval_samples if s.get("path")]
train_items = [(tr_map.get(p[0]) or tr_map.get(stem(p[0])), p[1]) for p in train_pairs]
train_items = [x for x in train_items if x[0]]
print(f"extracted in {(time.time()-t0)/60:.1f} min | train items with image: {len(train_items)} | eval samples: {len(eval_samples)}")
assert len(eval_samples) == CFG.n_eval, "some eval images were not found in Images_val.zip"

wl = np.array([len(c.split()) for _, c in train_items])
print(f"train caption words: mean {wl.mean():.1f}, median {np.median(wl):.0f}, p99 {np.percentile(wl,99):.0f}")
MAX_NEW_TOKENS = int(min(256, np.percentile(wl, 99) * 1.7))
print("generation max_new_tokens:", MAX_NEW_TOKENS)

['.gitattributes', 'Annotations_train.zip', 'Annotations_val.zip', 'Images_train.zip', 'Images_val.zip', 'README.md', 'VRSBench_EVAL_Cap.json', 'VRSBench_EVAL_referring.json', 'VRSBench_EVAL_vqa.json', 'VRSBench_train.json', 'fig_example.png']


VRSBench_EVAL_Cap.json: 0.00B [00:00, ?B/s]

EVAL_Cap records: 9350
sample record: {
 "image_id": "P0003_0002.png",
 "ground_truth": "The aerial image from GoogleEarth features a group of large yellow buses lined up on the left side, and two small vehicles, one on the top-right and another large vehicle on the bottom-right side of the image. The buses are parked in a lot adjacent to a green area and a road running roughly north-south.",
 "question": "Describe the image in detail",
 "dataset": "RSBench",
 "question_id": 0,
 "type": "caption"
}
unique eval images: 9350 | using: 50


VRSBench_train.json:   0%|          | 0.00/64.9M [00:00<?, ?B/s]

train json records: 142390
sample record: {
 "id": "Final_Data/v1.2",
 "image": "00002_0000.png",
 "conversations": [
  {
   "from": "human",
   "value": "<image>\n[caption] Could you describe the contents of this image for me?"
  },
  {
   "from": "gpt",
   "value": "The image, sourced from GoogleEarth, shows a rural area with an expressway-toll-station situated at the center. Alongside the toll station, multiple small vehicles are visible, with one positioned on the middle-left and another that is the right-most in the center relative to the toll station."
  }
 ]
}
caption training pairs: 20262 | train/eval overlap: 0


Images_val.zip:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

Images_val.zip:   0%|          | 0/9351 [00:00<?, ?it/s]

Images_train.zip:   0%|          | 0.00/8.36G [00:00<?, ?B/s]

Images_train.zip:   0%|          | 0/20265 [00:00<?, ?it/s]

extracted in 3.2 min | train items with image: 20262 | eval samples: 50
train caption words: mean 53.0, median 52, p99 100
generation max_new_tokens: 170


## 3. Metric: BERT-BLEU4 (as defined in the report)

`BERT-BLEU4 = LP * exp( (1/4) * sum_{n=1..4} log P_n )`, `P_n = (1/|R_n|) * sum_{r in R_n} max_{c in C_n} cos(E(c), E(r))`, `LP = exp(-0.5 * |Lc - Lr| / Lr)`.

* **Token-level (headline):** the whole caption goes through `bert-base-uncased` once; an n-gram vector is the mean of its contextual token vectors.
* **MiniLM phrase-level (secondary):** each n-gram string is embedded independently with `all-MiniLM-L6-v2`.

Lengths `Lc`, `Lr` are word counts. The official hackathon evaluator may differ in tokenisation details, so compare *relative* gains (baseline vs fine-tuned) and treat the absolute value as indicative.

In [4]:
from transformers import AutoTokenizer, AutoModel

class BertBleu:
    def __init__(self, name="bert-base-uncased", device="cuda"):
        self.tok = AutoTokenizer.from_pretrained(name); self.m = AutoModel.from_pretrained(name).to(device).eval(); self.dev = device
    @torch.no_grad()
    def tok_emb(self, text):
        enc = self.tok(text, return_tensors="pt", truncation=True, max_length=512).to(self.dev)
        return self.m(**enc).last_hidden_state[0][1:-1]                     # drop [CLS] / [SEP]
    @staticmethod
    def grams(E, n):
        if E.shape[0] < n: return None
        c = torch.cumsum(F.pad(E, (0, 0, 1, 0)), 0)
        return F.normalize((c[n:] - c[:-n]) / n, dim=-1)                       # mean of n consecutive token vectors
    def score(self, cand, ref, N=4, alpha=0.5):
        Ec, Er = self.tok_emb(cand), self.tok_emb(ref); Ps = []
        for n in range(1, N + 1):
            gc_, gr_ = self.grams(Ec, n), self.grams(Er, n)
            Ps.append(0.0 if gc_ is None or gr_ is None else (gr_ @ gc_.T).max(1).values.mean().item())
        Lc, Lr = len(cand.split()), len(ref.split())
        LP = math.exp(-alpha * abs(Lc - Lr) / max(Lr, 1))
        return LP * math.exp(sum(math.log(max(p, 1e-8)) for p in Ps) / N), Ps, LP

class MiniLMBleu:
    def __init__(self, device="cuda"):
        from sentence_transformers import SentenceTransformer
        self.m = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
    @staticmethod
    def words(t): return re.findall(r"[a-z0-9]+", t.lower())
    def grams(self, t, n):
        w = self.words(t); return [" ".join(w[i:i + n]) for i in range(len(w) - n + 1)]
    def score(self, cand, ref, N=4, alpha=0.5):
        Ps = []
        for n in range(1, N + 1):
            gc_, gr_ = self.grams(cand, n), self.grams(ref, n)
            if not gc_ or not gr_: Ps.append(0.0); continue
            ec = self.m.encode(gc_, convert_to_tensor=True, normalize_embeddings=True)
            er = self.m.encode(gr_, convert_to_tensor=True, normalize_embeddings=True)
            Ps.append((er @ ec.T).max(1).values.mean().item())
        Lc, Lr = len(self.words(cand)), len(self.words(ref))
        LP = math.exp(-alpha * abs(Lc - Lr) / max(Lr, 1))
        return LP * math.exp(sum(math.log(max(p, 1e-8)) for p in Ps) / N), Ps, LP

bb, mini = BertBleu(), MiniLMBleu()
r0, r1 = eval_samples[0]["ref"], eval_samples[1]["ref"]
print("sanity | identical text:", round(bb.score(r0, r0)[0], 4), round(mini.score(r0, r0)[0], 4),
      "| two different captions:", round(bb.score(r1, r0)[0], 4), round(mini.score(r1, r0)[0], 4),
      "| 1-word caption:", round(bb.score('a', r0)[0], 4))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sanity | identical text: 1.0 1.0 | two different captions: 0.5827 0.3994 | 1-word caption: 0.0


## 4. Generation + evaluation helpers (and optional micro-benchmark data pipeline)

In [5]:
@torch.no_grad()
def generate_captions(model, tok, samples, max_new_tokens=None, **gen_kwargs):
    FastVisionModel.for_inference(model)
    preds, lats = [], []
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": CFG.prompt}]}]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True)
    for s in tqdm(samples, desc="generating"):
        img = Image.open(s["path"]).convert("RGB")
        inputs = tok(images=img, text=text, add_special_tokens=False, return_tensors="pt").to("cuda")
        torch.cuda.synchronize(); t = time.time()
        out = model.generate(**inputs, max_new_tokens=max_new_tokens or MAX_NEW_TOKENS, use_cache=True, do_sample=False, **gen_kwargs)
        torch.cuda.synchronize(); lats.append(time.time() - t)
        preds.append(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return preds, lats

def evaluate(model, tok, name, **gen_kwargs):
    preds, lats = generate_captions(model, tok, eval_samples, **gen_kwargs)
    rows = []
    for s, p in zip(eval_samples, preds):
        sc, Ps, LP = bb.score(p, s["ref"]); sc2, _, _ = mini.score(p, s["ref"])
        rows.append(dict(image=s["image"], ref=s["ref"], pred=p, bert_bleu4=sc, minilm_bleu4=sc2, LP=LP, P1=Ps[0], P2=Ps[1], P3=Ps[2], P4=Ps[3],
                         words_pred=len(p.split()), words_ref=len(s["ref"].split())))
    m = lambda k: float(np.mean([r[k] for r in rows]))
    res = dict(name=name, bert_bleu4=m("bert_bleu4"), bert_bleu4_std=float(np.std([r["bert_bleu4"] for r in rows])), minilm_bleu4=m("minilm_bleu4"),
               LP=m("LP"), P1=m("P1"), P2=m("P2"), P3=m("P3"), P4=m("P4"), avg_words_pred=m("words_pred"), avg_words_ref=m("words_ref"),
               sec_per_caption=float(np.mean(lats)), rows=rows)
    print(f"[{name}] BERT-BLEU4 = {res['bert_bleu4']:.4f} (+/- {res['bert_bleu4_std']:.3f}) | MiniLM-BLEU4 = {res['minilm_bleu4']:.4f} | "
          f"LP = {res['LP']:.3f} | words pred/ref = {res['avg_words_pred']:.1f}/{res['avg_words_ref']:.1f} | {res['sec_per_caption']:.2f}s/caption")
    return res

def show_examples(res, k=3):
    for r in res["rows"][:k]:
        print(f"\n{r['image']}  (score {r['bert_bleu4']:.3f})\n  REF : {r['ref']}\n  PRED: {r['pred']}")

In [6]:
# ---------------- model / data / trainer building blocks (shared by the microbench and the real run) ----------------
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from torch.optim.lr_scheduler import LambdaLR

def load_model(gc_mode):
    model, tok = FastVisionModel.from_pretrained(CFG.model_name, load_in_4bit=True, use_gradient_checkpointing=gc_mode)
    model = FastVisionModel.get_peft_model(
        model, finetune_vision_layers=CFG.lora_vision, finetune_language_layers=True, finetune_attention_modules=True, finetune_mlp_modules=True,
        r=CFG.lora_r, lora_alpha=CFG.lora_alpha, lora_dropout=0.0, bias="none", random_state=CFG.seed, use_rslora=False, loftq_config=None,
        use_gradient_checkpointing=gc_mode)
    return model, tok

def to_messages(img, caption):
    return {"messages": [
        {"role": "user", "content": [{"type": "text", "text": CFG.prompt}, {"type": "image", "image": img}]},
        {"role": "assistant", "content": [{"type": "text", "text": caption}]}]}

class LazyCaptionDS(torch.utils.data.Dataset):          # images are decoded on demand -> constant RAM
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, c = self.items[i]; return to_messages(Image.open(p).convert("RGB"), c)
train_ds = LazyCaptionDS(train_items)

def make_collator(model, tok):
    if CFG.train_on_responses_only:
        try:
            return UnslothVisionDataCollator(model, tok, train_on_responses_only=True,
                                             instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
        except TypeError:
            print("WARNING: this Unsloth version has no train_on_responses_only in the vision collator; training on all tokens.")
    return UnslothVisionDataCollator(model, tok)

# --- wall-clock bookkeeping, time-aware LR schedule, budget callback ---
class Clock:
    def __init__(self): self.base, self.t0 = 0.0, None
    def elapsed(self): return self.base + (time.time() - self.t0 if self.t0 else 0.0)
CLOCK = Clock()
SCHED = dict(budget=CFG.train_budget_sec, warmup=CFG.warmup_steps, min_ratio=CFG.min_lr_ratio)
CLOCK_FILE = os.path.join(OUT_DIR, "clock.json")

class TimedTrainer(SFTTrainer):
    # cosine LR driven by progress = max(elapsed/budget, step/total_steps) -> always reaches min LR exactly at the cutoff
    def create_scheduler(self, num_training_steps, optimizer=None):
        optimizer = optimizer if optimizer is not None else self.optimizer
        total = max(int(num_training_steps), 1)
        def lr_lambda(step):
            if step < SCHED["warmup"]: return (step + 1) / max(SCHED["warmup"], 1)
            p = min(1.0, max(CLOCK.elapsed() / SCHED["budget"], step / total))
            return SCHED["min_ratio"] + (1 - SCHED["min_ratio"]) * 0.5 * (1 + math.cos(math.pi * p))
        self.lr_scheduler = LambdaLR(optimizer, lr_lambda)
        return self.lr_scheduler

class TimeBudget(TrainerCallback):
    def _dump(self): json.dump({"elapsed": CLOCK.elapsed()}, open(CLOCK_FILE, "w"))
    def on_train_begin(self, args, state, control, **kw): CLOCK.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if CLOCK.elapsed() >= CFG.train_budget_sec:
            print(f"\n*** time budget reached ({CLOCK.elapsed()/3600:.2f} h) at step {state.global_step}: saving and stopping ***")
            control.should_save = True; control.should_training_stop = True
        return control
    def on_save(self, args, state, control, **kw): self._dump()
    def on_train_end(self, args, state, control, **kw): self._dump()

class ProgressPrinter(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs and state.global_step % 50 == 0:
            el = CLOCK.elapsed(); eff = args.per_device_train_batch_size * args.gradient_accumulation_steps
            print(f"step {state.global_step} | loss {logs['loss']:.4f} | lr {logs.get('learning_rate', 0):.2e} | {el/3600:.2f}h/{CFG.train_budget_sec/3600:.1f}h | "
                  f"{state.global_step*eff/max(el,1):.2f} samples/s | peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB")

class StepTimer(TrainerCallback):
    def __init__(self): self.t = []
    def on_step_end(self, args, state, control, **kw): torch.cuda.synchronize(); self.t.append(time.time())

def build_trainer(model, tok, ds, *, bs, ga, max_steps, lr, optim, output_dir, callbacks, budget, warmup, min_ratio, save_steps):
    SCHED.update(budget=budget, warmup=warmup, min_ratio=min_ratio)
    FastVisionModel.for_training(model)
    kw = dict(per_device_train_batch_size=bs, gradient_accumulation_steps=ga, max_steps=max_steps, learning_rate=lr,
              fp16=not is_bf16_supported(), bf16=is_bf16_supported(), logging_steps=10, optim=optim, weight_decay=0.01,
              lr_scheduler_type="linear", warmup_steps=0, seed=CFG.seed, output_dir=output_dir, report_to="none",
              remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True},
              dataloader_num_workers=CFG.num_workers, dataloader_pin_memory=True, max_grad_norm=1.0,
              save_strategy="steps", save_steps=save_steps, save_total_limit=2)
    try: args = SFTConfig(max_length=CFG.max_len, **kw)
    except TypeError: args = SFTConfig(max_seq_length=CFG.max_len, **kw)
    common = dict(model=model, train_dataset=ds, data_collator=make_collator(model, tok), args=args, callbacks=callbacks)
    try: return TimedTrainer(processing_class=tok, **common)
    except TypeError: return TimedTrainer(tokenizer=tok, **common)

In [7]:
# ---------------- OPTIONAL micro-benchmark: measured effect of memory/speed optimisations (needs ~15 min) ----------------
MICROBENCH = []
if CFG.run_microbench:
    GRID = [dict(name="no grad-ckpt, bs1, AdamW 32-bit", gc=False,     bs=1, optim="adamw_torch"),
            dict(name="Unsloth GC,  bs1, AdamW 8-bit",   gc="unsloth", bs=1, optim="adamw_8bit"),
            dict(name="Unsloth GC,  bs4, AdamW 8-bit",   gc="unsloth", bs=4, optim="adamw_8bit"),
            dict(name="Unsloth GC,  bs8, AdamW 8-bit",   gc="unsloth", bs=8, optim="adamw_8bit")]
    for g in GRID:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); row = dict(config=g["name"])
        try:
            m_, t_ = load_model(g["gc"]); timer = StepTimer()
            tr = build_trainer(m_, t_, train_ds, bs=g["bs"], ga=1, max_steps=12, lr=1e-9, optim=g["optim"], output_dir=os.path.join(OUT_DIR, "mb"),
                               callbacks=[timer], budget=1e9, warmup=1, min_ratio=1.0, save_steps=10**9)
            tr.train()
            dt = float(np.median(np.diff(timer.t)[2:]))
            row.update(sec_per_step=dt, samples_per_sec=g["bs"] / dt, peak_vram_gb=torch.cuda.max_memory_allocated() / 2**30)
        except torch.cuda.OutOfMemoryError: row.update(error="OOM")
        except Exception as e: row.update(error=repr(e)[:150])
        MICROBENCH.append(row); print(row)
        for v in ("tr", "m_", "t_"): globals().pop(v, None)
        gc.collect(); torch.cuda.empty_cache()
    json.dump(MICROBENCH, open(os.path.join(OUT_DIR, "microbench.json"), "w"), indent=1)
else:
    print("microbench skipped (set CFG.run_microbench = True to run it)")

microbench skipped (set CFG.run_microbench = True to run it)


## 5. Load model and evaluate the zero-shot baseline (same 50 samples)

In [8]:
model, tok = load_model(CFG.gc_mode)
n_params = sum(p.numel() for p in model.parameters()); n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total params {n_params/1e9:.2f}B | trainable (LoRA) {n_train/1e6:.1f}M ({100*n_train/n_params:.2f}%) | VRAM after load {torch.cuda.memory_allocated()/2**30:.1f} GB")
base_res = evaluate(model, tok, "zero-shot baseline")
show_examples(base_res, 2)

==((====))==  Unsloth 2026.9.12: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Unsloth: Offloading embeddings to RAM to save 1.16 GB.


[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.visual`: `get_input_embeddings` not auto‑handled for Qwen3VLVisionModel; please override in the subclass.. Falling back to pre-forward hook.


total params 5.56B | trainable (LoRA) 102.7M (1.85%) | VRAM after load 6.9 GB


generating:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:5317: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


[zero-shot baseline] BERT-BLEU4 = 0.3236 (+/- 0.125) | MiniLM-BLEU4 = 0.3033 | LP = 0.449 | words pred/ref = 136.7/54.0 | 27.56s/caption

P0970_0023.png  (score 0.251)
  REF : The high-resolution image from GoogleEarth shows a neighborhood with houses and their backyards. One of the backyards contains a small vehicle in the bottom-left position. Surrounding the houses, there are patches of green vegetation and trees, and one of the houses has a swimming pool.
  PRED: This high-resolution satellite or aerial image captures a residential property situated on a waterfront, viewed from directly above. The scene is dominated by a large, L-shaped house with a dark gray or brown roof, surrounded by a lush, green yard. A prominent feature is a rectangular, above-ground swimming pool with a light-colored deck, located on a sloped, paved patio area. The property is bordered by dense trees and shrubs, creating a natural, secluded feel.

To the left, a white vehicle is parked on a paved driveway n

## 6. Train (hard 8 h cutoff)

Sanity check of the label masking first, then the run. The time-aware scheduler and `TimeBudget` callback guarantee that training stops at `train_budget_sec` and that the LR has decayed to its floor by then.
If the session dies, simply re-run all cells: the newest checkpoint is picked up and the elapsed time already spent is restored from `clock.json`.

In [9]:
col = make_collator(model, tok)
b = col([train_ds[0], train_ds[1]])
lab = b["labels"]; print({k: tuple(v.shape) for k, v in b.items() if hasattr(v, "shape")})
print(f"label tokens used in loss: {(lab != -100).float().mean().item()*100:.1f}% of all tokens (should be ~ caption length only)")
del col, b, lab

eff_bs = CFG.per_device_bs * CFG.grad_accum
total_steps = CFG.epochs_cap * (len(train_ds) // eff_bs)
ckpts = sorted(glob.glob(os.path.join(OUT_DIR, "checkpoint-*")), key=lambda p: int(p.split("-")[-1]))
resume_from = ckpts[-1] if (CFG.resume and ckpts) else None
if resume_from and os.path.exists(CLOCK_FILE):
    CLOCK.base = json.load(open(CLOCK_FILE))["elapsed"]; print(f"resuming from {resume_from}; {CLOCK.base/3600:.2f} h already used")

trainer = build_trainer(model, tok, train_ds, bs=CFG.per_device_bs, ga=CFG.grad_accum, max_steps=total_steps, lr=CFG.lr, optim=CFG.optim,
                        output_dir=OUT_DIR, callbacks=[TimeBudget(), ProgressPrinter()], budget=CFG.train_budget_sec,
                        warmup=CFG.warmup_steps, min_ratio=CFG.min_lr_ratio, save_steps=CFG.save_steps)
torch.cuda.reset_peak_memory_stats()
print(f"effective batch {eff_bs} | planned steps (epoch cap) {total_steps} | budget {CFG.train_budget_sec/3600:.1f} h | notebook time so far {(time.time()-NB_T0)/60:.0f} min")
train_out = trainer.train(resume_from_checkpoint=resume_from)

train_elapsed = CLOCK.elapsed()
steps_done = trainer.state.global_step
log_history = list(trainer.state.log_history)
TRAIN_STATS = dict(steps=steps_done, samples_seen=steps_done * eff_bs, epochs=steps_done * eff_bs / len(train_ds), elapsed_h=train_elapsed / 3600,
                   samples_per_sec=steps_done * eff_bs / max(train_elapsed, 1), sec_per_step=train_elapsed / max(steps_done, 1),
                   peak_vram_gb=torch.cuda.max_memory_allocated() / 2**30, final_train_loss=train_out.training_loss)
print(json.dumps(TRAIN_STATS, indent=1))

ADAPTER_DIR = os.path.join(OUT_DIR, "adapter")
model.save_pretrained(ADAPTER_DIR); tok.save_pretrained(ADAPTER_DIR); print("saved LoRA adapter to", ADAPTER_DIR)

Unsloth: Model does not have a default image size - using 512
{'input_ids': (2, 338), 'attention_mask': (2, 338), 'mm_token_type_ids': (2, 338), 'pixel_values': (2048, 1536), 'image_grid_thw': (2, 3), 'labels': (2, 338)}
label tokens used in loss: 17.0% of all tokens (should be ~ caption length only)
Unsloth: not enough free memory for dataset tokenization workers (~1GB each); tokenizing in-process.
Unsloth: Model does not have a default image size - using 512
effective batch 16 | planned steps (epoch cap) 3798 | budget 8.0 h | notebook time so far 29 min


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,262 | Num Epochs = 3 | Total steps = 3,798
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 102,693,888 of 8,869,817,584 (1.16% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.646739
20,1.961380
30,1.642743
40,1.561470
50,1.501806
60,1.469228
70,1.457241
80,1.423620
90,1.420337
100,1.406003


step 50 | loss 1.5018 | lr 1.99e-04 | 0.31h/8.0h | 0.72 samples/s | peak 9.2 GB
step 100 | loss 1.4060 | lr 1.97e-04 | 0.60h/8.0h | 0.74 samples/s | peak 9.2 GB
step 150 | loss 1.3551 | lr 1.94e-04 | 0.89h/8.0h | 0.75 samples/s | peak 9.2 GB
step 200 | loss 1.3595 | lr 1.90e-04 | 1.18h/8.0h | 0.75 samples/s | peak 9.2 GB


...

step 1250 | loss 1.2806 | lr 1.33e-05 | 7.33h/8.0h | 0.76 samples/s | peak 9.3 GB
step 1300 | loss 1.2121 | lr 1.11e-05 | 7.62h/8.0h | 0.76 samples/s | peak 9.3 GB
step 1350 | loss 1.2128 | lr 1.01e-05 | 7.91h/8.0h | 0.76 samples/s | peak 9.3 GB

*** time budget reached (8.00 h) at step 1366: saving and stopping ***


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/vrsbench_run/checkpoint-1366/tokenizer_config.json.


{
 "steps": 1366,
 "samples_seen": 21856,
 "epochs": 1.0786694304609614,
 "elapsed_h": 8.004207787447505,
 "samples_per_sec": 0.7584899433310629,
 "sec_per_step": 21.09454468141363,
 "peak_vram_gb": 9.254355907440186,
 "final_train_loss": 1.3290472145834569
}


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/vrsbench_run/adapter/tokenizer_config.json.


saved LoRA adapter to /kaggle/working/vrsbench_run/adapter


## 7. Evaluate the fine-tuned model on the same 50 samples

In [10]:
del trainer; gc.collect(); torch.cuda.empty_cache()
ft_res = evaluate(model, tok, "fine-tuned (greedy)")
show_examples(ft_res, 3)

sweep = [ft_res]
if CFG.decode_sweep:      # note: choosing a decoding setup on 50 samples is mildly optimistic; report greedy as the headline if you want a strict number
    try: sweep.append(evaluate(model, tok, "fine-tuned (beam=3)", num_beams=3))
    except Exception as e: print("beam search skipped:", repr(e)[:200])
    try: sweep.append(evaluate(model, tok, "fine-tuned (greedy, rep-pen 1.05)", repetition_penalty=1.05))
    except Exception as e: print("repetition-penalty run skipped:", repr(e)[:200])
best = max(sweep, key=lambda r: r["bert_bleu4"])
print(f"\nbest decoding on the 50 samples: {best['name']} -> BERT-BLEU4 {best['bert_bleu4']:.4f}")

generating:   0%|          | 0/50 [00:00<?, ?it/s]

[fine-tuned (greedy)] BERT-BLEU4 = 0.6851 (+/- 0.075) | MiniLM-BLEU4 = 0.6139 | LP = 0.863 | words pred/ref = 52.2/54.0 | 10.07s/caption

P0970_0023.png  (score 0.671)
  REF : The high-resolution image from GoogleEarth shows a neighborhood with houses and their backyards. One of the backyards contains a small vehicle in the bottom-left position. Surrounding the houses, there are patches of green vegetation and trees, and one of the houses has a swimming pool.
  PRED: The high-resolution image from GoogleEarth shows a residential area with a large swimming pool and a small vehicle. The swimming pool is located in the middle-right of the image, surrounded by a deck and adjacent to a house with a gray roof. A small vehicle is parked in the bottom-left corner of the image, near the edge of the frame.

08634_0000.png  (score 0.770)
  REF : The image from GoogleEarth features a barren landscape with a wind turbine visible in the center. The terrain is arid, primarily brown with various shade

generating:   0%|          | 0/50 [00:00<?, ?it/s]

[fine-tuned (beam=3)] BERT-BLEU4 = 0.6708 (+/- 0.090) | MiniLM-BLEU4 = 0.6040 | LP = 0.844 | words pred/ref = 53.3/54.0 | 12.73s/caption


generating:   0%|          | 0/50 [00:00<?, ?it/s]

[fine-tuned (greedy, rep-pen 1.05)] BERT-BLEU4 = 0.6770 (+/- 0.099) | MiniLM-BLEU4 = 0.6116 | LP = 0.853 | words pred/ref = 58.8/54.0 | 11.14s/caption

best decoding on the 50 samples: fine-tuned (greedy) -> BERT-BLEU4 0.6851


## 8. Report: tables, loss curve, saved artefacts

In [11]:
import pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

summary = pd.DataFrame([{k: v for k, v in r.items() if k != "rows"} for r in [base_res] + sweep]).round(4)
print(summary[["name", "bert_bleu4", "bert_bleu4_std", "minilm_bleu4", "LP", "P1", "P2", "P3", "P4", "avg_words_pred", "avg_words_ref", "sec_per_caption"]].to_string(index=False))

gain = ft_res["bert_bleu4"] - base_res["bert_bleu4"]
print(f"\nBERT-BLEU4: baseline {base_res['bert_bleu4']:.4f} -> fine-tuned {ft_res['bert_bleu4']:.4f} ({gain:+.4f}, {100*gain/max(base_res['bert_bleu4'],1e-9):+.1f}% relative)")
print("reference points from the reports: Qwen2.5-VL-7B SFT 0.6841 (token-level), Qwen3-VL SFT 0.7993 (team report, own implementation)")

steps = [h["step"] for h in log_history if "loss" in h]; losses = [h["loss"] for h in log_history if "loss" in h]
lrs = [h.get("learning_rate") for h in log_history if "loss" in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(steps, losses); ax[0].set_title("train loss"); ax[0].set_xlabel("step"); ax[0].grid(alpha=.3)
ax[1].plot(steps, lrs); ax[1].set_title("learning rate (time-aware cosine)"); ax[1].set_xlabel("step"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=130); plt.close()

pd.DataFrame(base_res["rows"]).to_csv(os.path.join(OUT_DIR, "preds_baseline.csv"), index=False)
for r in sweep: pd.DataFrame(r["rows"]).to_csv(os.path.join(OUT_DIR, "preds_" + re.sub(r"[^a-z0-9]+", "_", r["name"].lower()) + ".csv"), index=False)
RESULTS = dict(config=vars(CFG), gpu=torch.cuda.get_device_name(0), train_stats=TRAIN_STATS, microbench=MICROBENCH,
               baseline={k: v for k, v in base_res.items() if k != "rows"}, finetuned=[{k: v for k, v in r.items() if k != "rows"} for r in sweep],
               total_notebook_hours=(time.time() - NB_T0) / 3600)
json.dump(RESULTS, open(os.path.join(OUT_DIR, "results.json"), "w"), indent=1, default=str)
print(f"\nsaved to {OUT_DIR}: adapter/, results.json, training_curves.png, preds_*.csv | notebook wall-clock {RESULTS['total_notebook_hours']:.2f} h")

                             name  bert_bleu4  bert_bleu4_std  minilm_bleu4     LP     P1     P2     P3     P4  avg_words_pred  avg_words_ref  sec_per_caption
               zero-shot baseline      0.3236          0.1246        0.3033 0.4492 0.7055 0.7113 0.7263 0.7401          136.70          54.02          27.5601
              fine-tuned (greedy)      0.6851          0.0752        0.6139 0.8634 0.7725 0.7868 0.8030 0.8167           52.22          54.02          10.0725
              fine-tuned (beam=3)      0.6708          0.0898        0.6040 0.8441 0.7728 0.7872 0.8031 0.8163           53.28          54.02          12.7335
fine-tuned (greedy, rep-pen 1.05)      0.6770          0.0986        0.6116 0.8526 0.7770 0.7888 0.8034 0.8162           58.80          54.02          11.1398

BERT-BLEU4: baseline 0.3236 -> fine-tuned 0.6851 (+0.3614, +111.7% relative)
reference points from the reports: Qwen2.5-VL-7B SFT 0.6841 (token-level), Qwen3-VL SFT 0.7993 (team report, own implementation

## 9. What to put in the Task-01 PPT / README

* **Baseline vs optimised**: zero-shot vs fine-tuned BERT-BLEU4 (both variants), caption length vs reference length (the LP term), per-n precision `P1..P4`.
* **Training efficiency table**: `results.json -> train_stats` (samples/s, sec/step, peak VRAM, epochs reached in 8 h) and, if you enabled it, the micro-benchmark table (no grad-ckpt vs Unsloth GC, batch 1/4/8, 32-bit vs 8-bit AdamW).
* **Design decisions**: QLoRA 4-bit, LoRA on vision+language, response-only loss, time-aware cosine schedule, lazy data loading, checkpoint/resume with budget accounting.
* **Honest limitations**: 50-sample evaluation has wide confidence intervals (see the std column); decoding choice was compared on the same 50 samples; absolute scores depend on the exact BERT-BLEU implementation of the official evaluator.
* Ablations worth a slide if you have time: LoRA rank (16 vs 32 vs 64), 4B vs 8B model, with/without vision-layer LoRA. Each can be run by changing one line in `CFG` and shortening `train_budget_sec`.